In [43]:
import os
import torch

# load all environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  For a RULER subtask:
```
  python lm_eval_script.py \
    -m meta-llama/Llama-3.1-8B-Instruct \
    -t ruler_vt \
    --limit 1 \
    -kc baseline \
    -vc baseline \
    --dump_full_kv_dir ./results/kv_dumps
```
  For a LongBench subtask:
```
  python lm_eval_script.py \
    -m meta-llama/Llama-3.1-8B-Instruct \
    -t longbench_qasper \
    --limit 1 \
    -kc baseline \
    -vc baseline \
    --dump_full_kv_dir ./results/kv_dumps
```

### Load KVs

In [44]:
kv_dump_dir = "./results/kv_dumps/meta-llama_Llama-3.1-8B-Instruct/niah_multiquery_raw_kv.pt"
kvs = torch.load(kv_dump_dir)
print(kvs.keys())

keys = kvs["keys"]
values = kvs["values"]
prompt_len = int(kvs.get("prompt_len", keys.shape[-2]))

keys.shape, values.shape, prompt_len

dict_keys(['task_name', 'input_ids', 'prompt_len', 'model_name', 'keys', 'values'])


(torch.Size([32, 1, 8, 3770, 128]), torch.Size([32, 1, 8, 3770, 128]), 3770)

In [45]:
from IPython.display import display

from utils.matrix_decomposition import (
    DECOMP_METHODS,
    decompose_grouped_xkv_to_segment_store,
    decompose_to_segment_store,
    reconstruct_segments,
)
from utils.segmentation import (
    build_cluster_segment_ranges,
    group_keys_by_cluster,
    group_sequences_by_cluster,
    kmeans_cluster_sequences,
)


def _ensure_layer_batched_keys(keys):
    if keys.dim() == 4:
        return keys.unsqueeze(1)
    if keys.dim() == 5:
        return keys
    raise ValueError(
        "Expected keys with shape [layers, heads, seq, dim] or "
        "[layers, batch, heads, seq, dim]."
    )

def _resolve_kmeans_dtype(kmeans_dtype):
    if isinstance(kmeans_dtype, str):
        try:
            kmeans_dtype = getattr(torch, kmeans_dtype)
        except AttributeError as exc:
            raise ValueError(f"Unknown kmeans dtype: {kmeans_dtype}") from exc
    if not isinstance(kmeans_dtype, torch.dtype):
        raise TypeError("kmeans_dtype must be a torch.dtype or dtype name.")
    return kmeans_dtype

def _get_cluster_count(seq_len, n_clusters, kmeans_cluster_size=None):
    if seq_len == 0:
        return 0
    if kmeans_cluster_size is not None:
        cluster_count = int(round(seq_len / kmeans_cluster_size))
    else:
        cluster_count = n_clusters
    return max(1, min(cluster_count, seq_len))

def _select_prefix(keys, prefix_end=None, local_window=0):
    seq_len = keys.size(-2)
    if prefix_end is None:
        prefix_end = seq_len
    if prefix_end < 0 or prefix_end > seq_len:
        raise ValueError(
            "prefix_end must be between 0 and the sequence length."
        )
    suffix_start = max(0, prefix_end - local_window)
    return keys[..., :suffix_start, :], suffix_start

def _validate_cluster_axis(cluster_axis):
    if cluster_axis not in {"rows", "cols"}:
        raise ValueError("cluster_axis must be 'rows' or 'cols'.")

def _group_features_and_ranges(features, assignments, n_clusters):
    grouped_features, _, _ = group_sequences_by_cluster(features, assignments)
    segment_ranges = build_cluster_segment_ranges(
        assignments,
        n_clusters=n_clusters,
    )
    return grouped_features, segment_ranges

def _cluster_items_and_ranges(
    features,
    n_clusters,
    kmeans_n_iter,
    kmeans_init,
    kmeans_dtype,
    kmeans_cluster_size=None,
    cluster_axis="rows",
):
    _validate_cluster_axis(cluster_axis)
    item_matrix = (
        features
        if cluster_axis == "rows"
        else features.transpose(1, 2).contiguous()
    )
    cluster_count = _get_cluster_count(
        item_matrix.size(1),
        n_clusters,
        kmeans_cluster_size=kmeans_cluster_size,
    )
    assignments = kmeans_cluster_sequences(
        item_matrix,
        n_clusters=cluster_count,
        n_iter=max(1, kmeans_n_iter),
        kmeans_init=kmeans_init,
        dtype=kmeans_dtype,
    )
    grouped_items, segment_ranges = _group_features_and_ranges(
        item_matrix,
        assignments,
        cluster_count,
    )
    return item_matrix, grouped_items, assignments, segment_ranges, cluster_count

def _scatter_from_grouped(grouped_features, segment_ranges):
    metrics = []
    for batch_idx, batch_ranges in enumerate(segment_ranges):
        batch_features = grouped_features[batch_idx]
        frob_sq = batch_features.pow(2).sum()
        scatter = batch_features.new_zeros(())
        for start_idx, end_idx in batch_ranges:
            cluster = batch_features[start_idx:end_idx]
            centroid = cluster.mean(dim=0, keepdim=True)
            scatter = scatter + (cluster - centroid).pow(2).sum()

        denom = float(frob_sq.item())
        scatter_value = float(scatter.item())
        metrics.append(
            {
                "J_kmeans": scatter_value,
                "eta": scatter_value / denom if denom > 0 else 0.0,
            }
        )
    return metrics

def _group_tensor_last_dim_by_cluster(tensor, assignments):
    if assignments.shape != (tensor.size(0), tensor.size(-1)):
        raise ValueError(
            f"Expected assignments shape {(tensor.size(0), tensor.size(-1))}, "
            f"got {tuple(assignments.shape)}."
        )
    permutation = torch.argsort(assignments, dim=-1)
    inverse_permutation = torch.empty_like(permutation)
    original_positions = (
        torch.arange(
            permutation.size(-1),
            device=permutation.device,
            dtype=permutation.dtype,
        )
        .unsqueeze(0)
        .expand_as(permutation)
    )
    inverse_permutation.scatter_(1, permutation, original_positions)
    gather_shape = (tensor.size(0),) + (1,) * (tensor.dim() - 2) + (tensor.size(-1),)
    gather_idx = permutation.view(gather_shape).expand_as(tensor)
    grouped_tensor = torch.gather(tensor, dim=tensor.dim() - 1, index=gather_idx)
    return grouped_tensor, permutation, inverse_permutation

def _restore_tensor_last_dim(grouped_tensor, inverse_permutation):
    gather_shape = (grouped_tensor.size(0),) + (1,) * (grouped_tensor.dim() - 2) + (grouped_tensor.size(-1),)
    gather_idx = inverse_permutation.view(gather_shape).expand_as(grouped_tensor)
    return torch.gather(
        grouped_tensor,
        dim=grouped_tensor.dim() - 1,
        index=gather_idx,
    )

def _get_decomposition(
    decomposition_method="svd",
    rank_selection="comp_ratio",
    comp_ratio=2.0,
    energy_threshold=0.95,
    decomp_n_iter=3,
    decomp_lr=1e-2,
):
    if decomposition_method not in DECOMP_METHODS:
        raise ValueError(
            f"Unknown decomposition_method: {decomposition_method}. "
            f"Available methods: {sorted(DECOMP_METHODS)}"
        )
    return DECOMP_METHODS[decomposition_method], {
        "rank_selection": rank_selection,
        "cr": comp_ratio,
        "energy_threshold": energy_threshold,
        "n_iter": decomp_n_iter,
        "lr": decomp_lr,
        "quantise_a": False,
        "quantise_b": False,
        "compressor_bits": 4,
    }

def _low_rank_recon_metrics(original, reconstructed):
    if original.shape != reconstructed.shape:
        raise ValueError(
            f"Shape mismatch: {original.shape} vs {reconstructed.shape}"
        )
    orig_flat = original.reshape(original.size(0), -1)
    recon_flat = reconstructed.reshape(reconstructed.size(0), -1)
    numer = (orig_flat - recon_flat).pow(2).sum(dim=-1).sqrt()
    denom = orig_flat.pow(2).sum(dim=-1).sqrt()
    metrics = []
    for idx in range(original.size(0)):
        denom_value = float(denom[idx].item())
        error_value = float(numer[idx].item())
        metrics.append(
            {
                "low_rank_recon_error": error_value,
                "relative_low_rank_recon_error": (
                    error_value / denom_value if denom_value > 0 else 0.0
                ),
            }
        )
    return metrics

def _merge_metric_lists(*metric_lists):
    num_items = len(metric_lists[0])
    return [
        {
            key: value
            for metric_dict in metric_dicts
            for key, value in metric_dict.items()
        }
        for metric_dicts in zip(*metric_lists)
    ]

def _reconstruct_lr_segments(
    grouped_tensor,
    segment_ranges,
    decompose_fn,
    decomp_kwargs,
    cluster_axis="rows",
):
    _validate_cluster_axis(cluster_axis)
    tensor_to_decompose = (
        grouped_tensor
        if cluster_axis == "rows"
        else grouped_tensor.transpose(-2, -1).contiguous()
    )
    layer_segments = decompose_to_segment_store(
        tensor_to_decompose,
        decompose_fn,
        segment_ranges=segment_ranges,
        **decomp_kwargs,
    )
    reconstructed = reconstruct_segments(
        layer_segments,
        tensor_to_decompose[..., :0, :],
    )
    return (
        reconstructed
        if cluster_axis == "rows"
        else reconstructed.transpose(-2, -1).contiguous()
    )

def _reconstruct_xkv_segments(grouped_tensor, segment_ranges, decomp_kwargs, cluster_axis="rows"):
    _validate_cluster_axis(cluster_axis)
    tensor_to_decompose = (
        grouped_tensor
        if cluster_axis == "rows"
        else grouped_tensor.transpose(-2, -1).contiguous()
    )
    grouped_segments = decompose_grouped_xkv_to_segment_store(
        tensor_to_decompose.unsqueeze(1),
        segment_ranges=segment_ranges,
        **decomp_kwargs,
    )
    reconstructed = reconstruct_segments(
        grouped_segments,
        tensor_to_decompose.unsqueeze(1)[..., :0, :],
    ).squeeze(1)
    return (
        reconstructed
        if cluster_axis == "rows"
        else reconstructed.transpose(-2, -1).contiguous()
    )


def analyze_kmeans_lrk(
    keys,
    n_clusters=8,
    kmeans_cluster_size=None,
    kmeans_n_iter=8,
    kmeans_init="infllm",
    kmeans_dtype=torch.float32,
    kmeans_mode="per_head",
    prefix_end=None,
    local_window=0,
    include_head_breakdown=False,
    decomposition_method="svd",
    rank_selection="comp_ratio",
    comp_ratio=2.0,
    energy_threshold=0.95,
    decomp_n_iter=3,
    decomp_lr=1e-2,
    cluster_axis="rows",
):
    keys = _ensure_layer_batched_keys(keys)
    kmeans_dtype = _resolve_kmeans_dtype(kmeans_dtype)
    _validate_cluster_axis(cluster_axis)

    if kmeans_mode not in {"concat_heads", "avg_heads", "per_head"}:
        raise ValueError(
            "kmeans_mode must be one of 'concat_heads', 'avg_heads', or "
            "'per_head'."
        )

    decompose_fn, decomp_kwargs = _get_decomposition(
        decomposition_method=decomposition_method,
        rank_selection=rank_selection,
        comp_ratio=comp_ratio,
        energy_threshold=energy_threshold,
        decomp_n_iter=decomp_n_iter,
        decomp_lr=decomp_lr,
    )

    results = []
    for layer_idx in range(keys.size(0)):
        prefix_keys, compressed_len = _select_prefix(
            keys[layer_idx],
            prefix_end=prefix_end,
            local_window=local_window,
        )
        batch_size, num_heads, seq_len, head_dim = prefix_keys.shape

        if kmeans_mode == "per_head":
            base_tensor = prefix_keys.reshape(
                batch_size * num_heads,
                seq_len,
                head_dim,
            )
            (
                _,
                grouped_items,
                assignments,
                segment_ranges,
                cluster_count,
            ) = _cluster_items_and_ranges(
                base_tensor,
                n_clusters,
                kmeans_n_iter,
                kmeans_init,
                kmeans_dtype,
                kmeans_cluster_size=kmeans_cluster_size,
                cluster_axis=cluster_axis,
            )
            if cluster_axis == "rows":
                grouped_target = grouped_items
            else:
                grouped_target, _, _ = _group_tensor_last_dim_by_cluster(
                    base_tensor,
                    assignments,
                )
            scatter_metrics = _scatter_from_grouped(
                grouped_items,
                segment_ranges,
            )
            reconstructed = _reconstruct_lr_segments(
                grouped_target,
                segment_ranges,
                decompose_fn,
                decomp_kwargs,
                cluster_axis=cluster_axis,
            )
            recon_metrics = _low_rank_recon_metrics(
                grouped_target,
                reconstructed,
            )
            metrics = _merge_metric_lists(scatter_metrics, recon_metrics)
            for flat_idx, metric in enumerate(metrics):
                batch_idx = flat_idx // num_heads
                head_idx = flat_idx % num_heads
                results.append(
                    {
                        "cache_type": "kmeans_lr",
                        "cluster_axis": cluster_axis,
                        "kmeans_mode": kmeans_mode,
                        "metric_scope": "head",
                        "layer_idx": layer_idx,
                        "batch_idx": batch_idx,
                        "head_idx": head_idx,
                        "seq_len": seq_len,
                        "compressed_len": compressed_len,
                        "cluster_count": cluster_count,
                        **metric,
                    }
                )
            continue

        if kmeans_mode == "avg_heads":
            token_features = prefix_keys.mean(dim=1)
        else:
            token_features = prefix_keys.transpose(1, 2).reshape(
                batch_size,
                seq_len,
                -1,
            )

        (
            _,
            grouped_items,
            assignments,
            segment_ranges,
            cluster_count,
        ) = _cluster_items_and_ranges(
            token_features,
            n_clusters,
            kmeans_n_iter,
            kmeans_init,
            kmeans_dtype,
            kmeans_cluster_size=kmeans_cluster_size,
            cluster_axis=cluster_axis,
        )

        if cluster_axis == "rows":
            grouped_target, _, _ = group_keys_by_cluster(prefix_keys, assignments)
            scatter_metrics = _scatter_from_grouped(
                grouped_items,
                segment_ranges,
            )
            reconstructed = _reconstruct_lr_segments(
                grouped_target,
                segment_ranges,
                decompose_fn,
                decomp_kwargs,
                cluster_axis=cluster_axis,
            )
            layer_recon_metrics = _low_rank_recon_metrics(
                grouped_target,
                reconstructed,
            )
            layer_metrics = _merge_metric_lists(scatter_metrics, layer_recon_metrics)
            for batch_idx, metric in enumerate(layer_metrics):
                results.append(
                    {
                        "cache_type": "kmeans_lr",
                        "cluster_axis": cluster_axis,
                        "kmeans_mode": kmeans_mode,
                        "metric_scope": "layer",
                        "layer_idx": layer_idx,
                        "batch_idx": batch_idx,
                        "head_idx": None,
                        "seq_len": seq_len,
                        "compressed_len": compressed_len,
                        "cluster_count": cluster_count,
                        **metric,
                    }
                )

            if include_head_breakdown:
                for head_idx in range(num_heads):
                    head_scatter_metrics = _scatter_from_grouped(
                        grouped_target[:, head_idx],
                        segment_ranges,
                    )
                    head_recon_metrics = _low_rank_recon_metrics(
                        grouped_target[:, head_idx],
                        reconstructed[:, head_idx],
                    )
                    head_metrics = _merge_metric_lists(
                        head_scatter_metrics,
                        head_recon_metrics,
                    )
                    for batch_idx, metric in enumerate(head_metrics):
                        results.append(
                            {
                                "cache_type": "kmeans_lr",
                                "cluster_axis": cluster_axis,
                                "kmeans_mode": kmeans_mode,
                                "metric_scope": "head",
                                "layer_idx": layer_idx,
                                "batch_idx": batch_idx,
                                "head_idx": head_idx,
                                "seq_len": seq_len,
                                "compressed_len": compressed_len,
                                "cluster_count": cluster_count,
                                **metric,
                            }
                        )
            continue

        grouped_target, _, inverse_permutation = _group_tensor_last_dim_by_cluster(
            token_features,
            assignments,
        )
        scatter_metrics = _scatter_from_grouped(
            grouped_items,
            segment_ranges,
        )
        reconstructed = _reconstruct_lr_segments(
            grouped_target,
            segment_ranges,
            decompose_fn,
            decomp_kwargs,
            cluster_axis=cluster_axis,
        )
        restored_reconstructed = _restore_tensor_last_dim(
            reconstructed,
            inverse_permutation,
        )
        layer_recon_metrics = _low_rank_recon_metrics(
            grouped_target,
            reconstructed,
        )
        layer_metrics = _merge_metric_lists(scatter_metrics, layer_recon_metrics)
        for batch_idx, metric in enumerate(layer_metrics):
            results.append(
                {
                    "cache_type": "kmeans_lr",
                    "cluster_axis": cluster_axis,
                    "kmeans_mode": kmeans_mode,
                    "metric_scope": "layer",
                    "layer_idx": layer_idx,
                    "batch_idx": batch_idx,
                    "head_idx": None,
                    "seq_len": seq_len,
                    "compressed_len": compressed_len,
                    "cluster_count": cluster_count,
                    **metric,
                }
            )

        if include_head_breakdown:
            for head_idx in range(num_heads):
                if kmeans_mode == "avg_heads":
                    head_target = prefix_keys[:, head_idx]
                    head_assignments = assignments
                    grouped_head_target, _, _ = _group_tensor_last_dim_by_cluster(
                        head_target,
                        head_assignments,
                    )
                    grouped_head_items, head_segment_ranges = _group_features_and_ranges(
                        head_target.transpose(1, 2).contiguous(),
                        head_assignments,
                        cluster_count,
                    )
                    head_reconstructed = _reconstruct_lr_segments(
                        grouped_head_target,
                        head_segment_ranges,
                        decompose_fn,
                        decomp_kwargs,
                        cluster_axis=cluster_axis,
                    )
                    head_recon_metrics = _low_rank_recon_metrics(
                        grouped_head_target,
                        head_reconstructed,
                    )
                    head_scatter_metrics = _scatter_from_grouped(
                        grouped_head_items,
                        head_segment_ranges,
                    )
                else:
                    start_idx = head_idx * head_dim
                    end_idx = start_idx + head_dim
                    head_items, head_segment_ranges = _group_features_and_ranges(
                        token_features[..., start_idx:end_idx]
                        .transpose(1, 2)
                        .contiguous(),
                        assignments[..., start_idx:end_idx],
                        cluster_count,
                    )
                    head_scatter_metrics = _scatter_from_grouped(
                        head_items,
                        head_segment_ranges,
                    )
                    head_recon_metrics = _low_rank_recon_metrics(
                        token_features[..., start_idx:end_idx],
                        restored_reconstructed[..., start_idx:end_idx],
                    )

                head_metrics = _merge_metric_lists(
                    head_scatter_metrics,
                    head_recon_metrics,
                )
                for batch_idx, metric in enumerate(head_metrics):
                    results.append(
                        {
                            "cache_type": "kmeans_lr",
                            "cluster_axis": cluster_axis,
                            "kmeans_mode": kmeans_mode,
                            "metric_scope": "head",
                            "layer_idx": layer_idx,
                            "batch_idx": batch_idx,
                            "head_idx": head_idx,
                            "seq_len": seq_len,
                            "compressed_len": compressed_len,
                            "cluster_count": cluster_count,
                            **metric,
                        }
                    )

    return results


def _get_group_bounds(layer_idx, layer_group_size, num_layers=None):
    group_start = (layer_idx // layer_group_size) * layer_group_size
    group_last = group_start + layer_group_size - 1
    if num_layers is not None:
        group_last = min(group_last, num_layers - 1)
    return group_start, group_last


def analyze_kmeans_xkv(
    keys,
    layer_group_size=2,
    num_layers=None,
    n_clusters=8,
    kmeans_cluster_size=None,
    kmeans_n_iter=8,
    kmeans_init="infllm",
    kmeans_dtype=torch.float32,
    prefix_end=None,
    local_window=0,
    decomposition_method="svd",
    rank_selection="comp_ratio",
    comp_ratio=2.0,
    energy_threshold=0.95,
    decomp_n_iter=3,
    decomp_lr=1e-2,
    cluster_axis="rows",
):
    keys = _ensure_layer_batched_keys(keys)
    kmeans_dtype = _resolve_kmeans_dtype(kmeans_dtype)
    _validate_cluster_axis(cluster_axis)

    if layer_group_size <= 0:
        raise ValueError("layer_group_size must be positive.")
    if num_layers is None:
        num_layers = keys.size(0)
    if num_layers <= 0 or num_layers > keys.size(0):
        raise ValueError("num_layers must be in [1, keys.size(0)].")

    if decomposition_method != "svd":
        raise NotImplementedError(
            "KMeansXKVKeysCache-style grouped reconstruction currently "
            "supports decomposition_method='svd' only."
        )

    _, decomp_kwargs = _get_decomposition(
        decomposition_method=decomposition_method,
        rank_selection=rank_selection,
        comp_ratio=comp_ratio,
        energy_threshold=energy_threshold,
        decomp_n_iter=decomp_n_iter,
        decomp_lr=decomp_lr,
    )

    results = []
    for layer_idx in range(num_layers):
        group_start, group_last = _get_group_bounds(
            layer_idx,
            layer_group_size,
            num_layers,
        )
        if layer_idx != group_last:
            continue

        group_tensors = [keys[i] for i in range(group_start, group_last + 1)]
        seq_len = group_tensors[-1].size(-2)
        if any(tensor.size(-2) != seq_len for tensor in group_tensors[:-1]):
            raise ValueError(
                "All layers in an xKV group must share the same cached length."
            )

        prefix_tensors = []
        split_sizes = []
        compressed_len = None
        for tensor in group_tensors:
            prefix_tensor, compressed_len = _select_prefix(
                tensor,
                prefix_end=prefix_end,
                local_window=local_window,
            )
            prefix_flat = prefix_tensor.transpose(1, 2).reshape(
                prefix_tensor.size(0),
                prefix_tensor.size(-2),
                -1,
            )
            prefix_tensors.append(prefix_flat)
            split_sizes.append(prefix_flat.size(-1))

        group_prefix = torch.cat(prefix_tensors, dim=-1)
        (
            _,
            grouped_items,
            assignments,
            segment_ranges,
            cluster_count,
        ) = _cluster_items_and_ranges(
            group_prefix,
            n_clusters,
            kmeans_n_iter,
            kmeans_init,
            kmeans_dtype,
            kmeans_cluster_size=kmeans_cluster_size,
            cluster_axis=cluster_axis,
        )

        if cluster_axis == "rows":
            grouped_target = grouped_items
            reconstructed_group = _reconstruct_xkv_segments(
                grouped_target,
                segment_ranges,
                decomp_kwargs,
                cluster_axis=cluster_axis,
            )
            grouped_layer_targets = torch.split(
                grouped_target,
                split_sizes,
                dim=-1,
            )
            reconstructed_layer_targets = torch.split(
                reconstructed_group,
                split_sizes,
                dim=-1,
            )
        else:
            grouped_target, _, inverse_permutation = _group_tensor_last_dim_by_cluster(
                group_prefix,
                assignments,
            )
            reconstructed_group = _reconstruct_xkv_segments(
                grouped_target,
                segment_ranges,
                decomp_kwargs,
                cluster_axis=cluster_axis,
            )
            restored_reconstructed = _restore_tensor_last_dim(
                reconstructed_group,
                inverse_permutation,
            )

        group_scatter_metrics = _scatter_from_grouped(
            grouped_items,
            segment_ranges,
        )
        group_recon_metrics = _low_rank_recon_metrics(
            grouped_target,
            reconstructed_group,
        )
        group_metrics = _merge_metric_lists(
            group_scatter_metrics,
            group_recon_metrics,
        )
        for batch_idx, metric in enumerate(group_metrics):
            results.append(
                {
                    "cache_type": "kmeans_xkv",
                    "cluster_axis": cluster_axis,
                    "metric_scope": "group",
                    "layer_idx": group_last,
                    "group_start_layer": group_start,
                    "group_last_layer": group_last,
                    "batch_idx": batch_idx,
                    "head_idx": None,
                    "seq_len": group_prefix.size(1),
                    "compressed_len": compressed_len,
                    "cluster_count": cluster_count,
                    **metric,
                }
            )

        col_offset = 0
        for offset, split_size in enumerate(split_sizes):
            actual_layer_idx = group_start + offset
            if cluster_axis == "rows":
                layer_target = grouped_layer_targets[offset]
                layer_recon = reconstructed_layer_targets[offset]
                layer_scatter_metrics = _scatter_from_grouped(
                    layer_target,
                    segment_ranges,
                )
                layer_recon_metrics = _low_rank_recon_metrics(
                    layer_target,
                    layer_recon,
                )
            else:
                layer_original = group_prefix[..., col_offset : col_offset + split_size]
                layer_assignments = assignments[
                    ..., col_offset : col_offset + split_size
                ]
                layer_items, layer_segment_ranges = _group_features_and_ranges(
                    layer_original.transpose(1, 2).contiguous(),
                    layer_assignments,
                    cluster_count,
                )
                layer_scatter_metrics = _scatter_from_grouped(
                    layer_items,
                    layer_segment_ranges,
                )
                layer_recon_metrics = _low_rank_recon_metrics(
                    layer_original,
                    restored_reconstructed[
                        ..., col_offset : col_offset + split_size
                    ],
                )
                col_offset += split_size

            layer_metrics = _merge_metric_lists(
                layer_scatter_metrics,
                layer_recon_metrics,
            )
            for batch_idx, metric in enumerate(layer_metrics):
                results.append(
                    {
                        "cache_type": "kmeans_xkv",
                        "cluster_axis": cluster_axis,
                        "metric_scope": "layer",
                        "layer_idx": actual_layer_idx,
                        "group_start_layer": group_start,
                        "group_last_layer": group_last,
                        "batch_idx": batch_idx,
                        "head_idx": None,
                        "seq_len": group_prefix.size(1),
                        "compressed_len": compressed_len,
                        "cluster_count": cluster_count,
                        **metric,
                    }
                )

    return results


### Cluster

In [46]:
kmeans_cfg = {
    "n_clusters": 8,
    "kmeans_cluster_size": None,
    "kmeans_n_iter": 8,
    "kmeans_init": "infllm",
    "kmeans_dtype": torch.float32,
}

decomposition_cfg = {
    "decomposition_method": "svd",
    "rank_selection": "comp_ratio",
    "comp_ratio": 2.0,
    "energy_threshold": 0.95,
    "decomp_n_iter": 3,
    "decomp_lr": 1e-2,
}

lrk_mode = "per_head"  # "avg_heads" or "per_head"
cluster_axis = "cols"  # "rows" or "cols"
include_lrk_head_breakdown = False
prefix_end = prompt_len
local_window = 0

xkv_layer_group_size = 4
xkv_num_layers = keys.size(0)


In [47]:
lrk_results = analyze_kmeans_lrk(
    keys,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg,
    **decomposition_cfg,
)

xkv_results = analyze_kmeans_xkv(
    keys,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg,
    **decomposition_cfg,
)

kmeans_cfg_n1 = {**kmeans_cfg, "n_clusters": 1, "kmeans_cluster_size": None}

lrk_results_n1 = analyze_kmeans_lrk(
    keys,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

xkv_results_n1 = analyze_kmeans_xkv(
    keys,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("First LRK result:", lrk_results[0] if lrk_results else None)
    print("First xKV result:", xkv_results[0] if xkv_results else None)
    print("First LRK n=1 result:", lrk_results_n1[0] if lrk_results_n1 else None)
    print("First xKV n=1 result:", xkv_results_n1[0] if xkv_results_n1 else None)
else:
    cols_to_hide = ["batch_idx", "seq_len", "compressed_len"]
    lrk_df = pd.DataFrame(lrk_results).drop(columns=cols_to_hide, errors="ignore")
    xkv_df = pd.DataFrame(xkv_results).drop(columns=cols_to_hide, errors="ignore")
    lrk_df_n1 = pd.DataFrame(lrk_results_n1).drop(columns=cols_to_hide, errors="ignore")
    xkv_df_n1 = pd.DataFrame(xkv_results_n1).drop(columns=cols_to_hide, errors="ignore")

    def build_summary(label, df):
        return {
            "case": label,
            "rows": len(df),
            "mean_eta": df["eta"].mean(),
            "median_eta": df["eta"].median(),
            "mean_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].mean(),
            "median_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].median(),
            "min_eta": df["eta"].min(),
            "max_eta": df["eta"].max(),
        }

    summary_df = pd.DataFrame(
        [
            build_summary(f"lrk_{cluster_axis}", lrk_df),
            build_summary(f"xkv_{cluster_axis}", xkv_df),
            build_summary(f"lrk_{cluster_axis}_n_clusters_1", lrk_df_n1),
            build_summary(f"xkv_{cluster_axis}_n_clusters_1", xkv_df_n1),
        ]
    )
    display(summary_df)
    display(lrk_df)
    display(xkv_df)
    display(lrk_df_n1)
    display(xkv_df_n1)


,case,rows,mean_eta,median_eta,mean_relative_low_rank_recon_error,median_relative_low_rank_recon_error,min_eta,max_eta
0,lrk_cols,256,0.527533,0.526304,0.313961,0.319358,0.224684,0.802920
1,xkv_cols,40,0.924115,0.982470,0.127327,0.129057,0.652318,0.993464
2,lrk_cols_n_clusters_1,256,0.992013,0.993289,0.266784,0.273068,0.970588,1.000000
3,xkv_cols_n_clusters_1,40,0.999300,1.000000,0.125397,0.127698,0.992481,1.000000


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,cols,per_head,head,0,0,8,757760.0,0.502717,374.0,0.305556
1,kmeans_lr,cols,per_head,head,0,1,8,561152.0,0.475694,316.0,0.290441
2,kmeans_lr,cols,per_head,head,0,2,8,745472.0,0.694656,306.0,0.296512
3,kmeans_lr,cols,per_head,head,0,3,8,360448.0,0.530120,222.0,0.268116
4,kmeans_lr,cols,per_head,head,0,4,8,313344.0,0.466463,218.0,0.265854
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,cols,per_head,head,31,3,8,1720320.0,0.600000,408.0,0.240566
252,kmeans_lr,cols,per_head,head,31,4,8,819200.0,0.502513,450.0,0.351562
253,kmeans_lr,cols,per_head,head,31,5,8,1032192.0,0.580645,450.0,0.336826
254,kmeans_lr,cols,per_head,head,31,6,8,1130496.0,0.448052,338.0,0.212312


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,cols,group,3,0,3,None,8,58195968.0,0.965217,936.0,0.120370
1,kmeans_xkv,cols,layer,0,0,3,None,8,8126464.0,0.961240,414.0,0.142170
2,kmeans_xkv,cols,layer,1,0,3,None,8,16777216.0,0.941176,414.0,0.098011
3,kmeans_xkv,cols,layer,2,0,3,None,8,17301504.0,0.985075,478.0,0.114027
4,kmeans_xkv,cols,layer,3,0,3,None,8,16056320.0,0.976096,552.0,0.136364
5,kmeans_xkv,cols,group,7,4,7,None,8,76021760.0,0.986395,1128.0,0.128650
6,kmeans_xkv,cols,layer,4,4,7,None,8,17039360.0,0.977444,560.0,0.134615
7,kmeans_xkv,cols,layer,5,4,7,None,8,18087936.0,0.985714,556.0,0.129664
8,kmeans_xkv,cols,layer,6,4,7,None,8,19398656.0,0.986667,572.0,0.128597
9,kmeans_xkv,cols,layer,7,4,7,None,8,21364736.0,0.987879,568.0,0.121575


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,cols,per_head,head,0,0,1,1490944.0,0.989130,234.0,0.191176
1,kmeans_lr,cols,per_head,head,0,1,1,1146880.0,0.972222,204.0,0.187500
2,kmeans_lr,cols,per_head,head,0,2,1,1064960.0,0.992366,253.0,0.245155
3,kmeans_lr,cols,per_head,head,0,3,1,679936.0,1.000000,142.0,0.171498
4,kmeans_lr,cols,per_head,head,0,4,1,663552.0,0.987805,97.5,0.118902
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,cols,per_head,head,31,3,1,2818048.0,0.982857,366.0,0.215802
252,kmeans_lr,cols,per_head,head,31,4,1,1605632.0,0.984925,382.0,0.298438
253,kmeans_lr,cols,per_head,head,31,5,1,1761280.0,0.990783,400.0,0.299401
254,kmeans_lr,cols,per_head,head,31,6,1,2506752.0,0.993506,308.0,0.193467


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,cols,group,3,0,3,None,1,60293120.0,1.000000,920.0,0.118313
1,kmeans_xkv,cols,layer,0,0,3,None,1,8454144.0,1.000000,408.0,0.140110
2,kmeans_xkv,cols,layer,1,0,3,None,1,17825792.0,1.000000,394.0,0.093277
3,kmeans_xkv,cols,layer,2,0,3,None,1,17563648.0,1.000000,478.0,0.114027
4,kmeans_xkv,cols,layer,3,0,3,None,1,16449536.0,1.000000,548.0,0.135375
5,kmeans_xkv,cols,group,7,4,7,None,1,77070336.0,1.000000,1128.0,0.128650
6,kmeans_xkv,cols,layer,4,4,7,None,1,17301504.0,0.992481,560.0,0.134615
7,kmeans_xkv,cols,layer,5,4,7,None,1,18350080.0,1.000000,552.0,0.128731
8,kmeans_xkv,cols,layer,6,4,7,None,1,19660800.0,1.000000,572.0,0.128597
9,kmeans_xkv,cols,layer,7,4,7,None,1,21626880.0,1.000000,564.0,0.120719


In [ ]:
lrk_results = analyze_kmeans_lrk(
    values,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg,
    **decomposition_cfg,
)

xkv_results = analyze_kmeans_xkv(
    values,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg,
    **decomposition_cfg,
)

kmeans_cfg_n1 = {**kmeans_cfg, "n_clusters": 1, "kmeans_cluster_size": None}

lrk_results_n1 = analyze_kmeans_lrk(
    values,
    kmeans_mode=lrk_mode,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    include_head_breakdown=include_lrk_head_breakdown,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

xkv_results_n1 = analyze_kmeans_xkv(
    values,
    layer_group_size=xkv_layer_group_size,
    num_layers=xkv_num_layers,
    prefix_end=prefix_end,
    local_window=local_window,
    cluster_axis=cluster_axis,
    **kmeans_cfg_n1,
    **decomposition_cfg,
)

try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("First LRK result:", lrk_results[0] if lrk_results else None)
    print("First xKV result:", xkv_results[0] if xkv_results else None)
    print("First LRK n=1 result:", lrk_results_n1[0] if lrk_results_n1 else None)
    print("First xKV n=1 result:", xkv_results_n1[0] if xkv_results_n1 else None)
else:
    cols_to_hide = ["batch_idx", "seq_len", "compressed_len"]
    lrk_df = pd.DataFrame(lrk_results).drop(columns=cols_to_hide, errors="ignore")
    xkv_df = pd.DataFrame(xkv_results).drop(columns=cols_to_hide, errors="ignore")
    lrk_df_n1 = pd.DataFrame(lrk_results_n1).drop(columns=cols_to_hide, errors="ignore")
    xkv_df_n1 = pd.DataFrame(xkv_results_n1).drop(columns=cols_to_hide, errors="ignore")

    def build_summary(label, df):
        return {
            "case": label,
            "rows": len(df),
            "mean_eta": df["eta"].mean(),
            "median_eta": df["eta"].median(),
            "mean_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].mean(),
            "median_relative_low_rank_recon_error": df[
                "relative_low_rank_recon_error"
            ].median(),
            "min_eta": df["eta"].min(),
            "max_eta": df["eta"].max(),
        }

    summary_df = pd.DataFrame(
        [
            build_summary(f"lrk_{cluster_axis}", lrk_df),
            build_summary(f"xkv_{cluster_axis}", xkv_df),
            build_summary(f"lrk_{cluster_axis}_n_clusters_1", lrk_df_n1),
            build_summary(f"xkv_{cluster_axis}_n_clusters_1", xkv_df_n1),
        ]
    )
    display(summary_df)
    display(lrk_df)
    display(xkv_df)
    display(lrk_df_n1)
    display(xkv_df_n1)

,case,rows,mean_eta,median_eta,min_eta,max_eta
0,lrk,256,0.782432,0.807589,0.138393,0.936975
1,xkv,40,0.820058,0.837185,0.466837,0.895062
2,lrk_n_clusters_1,256,0.890968,0.922806,0.213010,0.988571
3,xkv_n_clusters_1,40,0.883587,0.905339,0.528061,0.939560


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,rows,per_head,head,0,0,8,584.0,0.784946,5.18750,0.190367
1,kmeans_lr,rows,per_head,head,0,1,8,708.0,0.917098,11.81250,0.425676
2,kmeans_lr,rows,per_head,head,0,2,8,498.0,0.876761,6.90625,0.289267
3,kmeans_lr,rows,per_head,head,0,3,8,438.0,0.862205,8.37500,0.372222
4,kmeans_lr,rows,per_head,head,0,4,8,406.0,0.890351,7.06250,0.330409
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,rows,per_head,head,31,3,8,111104.0,0.138393,138.00000,0.154018
252,kmeans_lr,rows,per_head,head,31,4,8,66048.0,0.767857,114.50000,0.389456
253,kmeans_lr,rows,per_head,head,31,5,8,145408.0,0.865854,176.00000,0.429268
254,kmeans_lr,rows,per_head,head,31,6,8,101376.0,0.664430,138.00000,0.353846


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,rows,group,3,0,3,None,8,246784.0,0.836806,109.50,0.201287
1,kmeans_xkv,rows,layer,0,0,3,None,8,4256.0,0.801205,23.25,0.318493
2,kmeans_xkv,rows,layer,1,0,3,None,8,19712.0,0.850829,36.00,0.236842
3,kmeans_xkv,rows,layer,2,0,3,None,8,84992.0,0.809756,65.50,0.202160
4,kmeans_xkv,rows,layer,3,0,3,None,8,137216.0,0.848101,76.50,0.190299
5,kmeans_xkv,rows,group,7,4,7,None,8,581632.0,0.820809,243.00,0.289286
6,kmeans_xkv,rows,layer,4,4,7,None,8,124416.0,0.832192,121.00,0.313472
7,kmeans_xkv,rows,layer,5,4,7,None,8,123392.0,0.792763,112.50,0.285533
8,kmeans_xkv,rows,layer,6,4,7,None,8,152576.0,0.846591,124.50,0.293632
9,kmeans_xkv,rows,layer,7,4,7,None,8,183296.0,0.817352,127.00,0.267932


,cache_type,cluster_axis,kmeans_mode,metric_scope,layer_idx,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_lr,rows,per_head,head,0,0,1,712.0,0.956989,9.4375,0.346330
1,kmeans_lr,rows,per_head,head,0,1,1,752.0,0.974093,14.3750,0.518018
2,kmeans_lr,rows,per_head,head,0,2,1,560.0,0.985915,8.9375,0.374346
3,kmeans_lr,rows,per_head,head,0,3,1,488.0,0.960630,10.5625,0.469444
4,kmeans_lr,rows,per_head,head,0,4,1,446.0,0.978070,8.1875,0.383041
...,...,...,...,...,...,...,...,...,...,...,...
251,kmeans_lr,rows,per_head,head,31,3,1,171008.0,0.213010,153.0000,0.170759
252,kmeans_lr,rows,per_head,head,31,4,1,77824.0,0.904762,133.0000,0.452381
253,kmeans_lr,rows,per_head,head,31,5,1,156672.0,0.932927,210.0000,0.512195
254,kmeans_lr,rows,per_head,head,31,6,1,123904.0,0.812081,154.0000,0.394872


,cache_type,cluster_axis,metric_scope,layer_idx,group_start_layer,group_last_layer,head_idx,cluster_count,J_kmeans,eta,low_rank_recon_error,relative_low_rank_recon_error
0,kmeans_xkv,rows,group,3,0,3,None,1,270336.0,0.916667,95.500,0.175551
1,kmeans_xkv,rows,layer,0,0,3,None,1,4672.0,0.879518,26.875,0.368151
2,kmeans_xkv,rows,layer,1,0,3,None,1,21888.0,0.939560,39.500,0.258170
3,kmeans_xkv,rows,layer,2,0,3,None,1,92672.0,0.882927,58.500,0.180556
4,kmeans_xkv,rows,layer,3,0,3,None,1,150528.0,0.930380,58.750,0.146144
5,kmeans_xkv,rows,group,7,4,7,None,1,651264.0,0.919075,194.000,0.230952
6,kmeans_xkv,rows,layer,4,4,7,None,1,136192.0,0.910959,101.000,0.261658
7,kmeans_xkv,rows,layer,5,4,7,None,1,140288.0,0.901316,94.500,0.239848
8,kmeans_xkv,rows,layer,6,4,7,None,1,167936.0,0.931818,96.500,0.227594
9,kmeans_xkv,rows,layer,7,4,7,None,1,204800.0,0.913242,95.500,0.201477
